In [3]:
# [환경 설정] 한글 폰트 설정 및 필수 라이브러리 로드
# macOS, Windows, Linux, Google Colab 환경에 맞춰 한글 깨짐 없이 동작하도록 자동 감지 설정합니다.

import platform
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pandas as pd
import numpy as np

try:
    import seaborn as sns
except ImportError:
    pass

# 운영체제(OS)별 한글 폰트 자동 설정
system_name = platform.system()
if system_name == 'Darwin':          # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
    plt.rcParams['font.sans-serif'] = ['AppleGothic', 'Apple SD Gothic Neo', 'NanumGothic', 'DejaVu Sans']
elif system_name == 'Windows':       # Windows
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['font.sans-serif'] = ['Malgun Gothic', 'NanumGothic', 'DejaVu Sans']
else:                               # Linux / Google Colab
    try:
        nanum_fonts = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name]
        if nanum_fonts:
            plt.rcParams['font.family'] = nanum_fonts[0]
        else:
            import subprocess
            subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], check=False, stdout=subprocess.DEVNULL)
            fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
            plt.rcParams['font.family'] = 'NanumGothic'
    except Exception:
        pass
    plt.rcParams['font.sans-serif'] = ['NanumGothic', 'DejaVu Sans']

# 마이너스 기호 깨짐 방지 및 Seaborn 폰트 동기화
plt.rcParams['axes.unicode_minus'] = False
try:
    if 'sns' in locals():
        sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'])
except Exception:
    pass

print(f'✅ 환경 설정 완료! 현재 적용된 폰트: {plt.rcParams["font.family"]}')

✅ 환경 설정 완료! 현재 적용된 폰트: ['Malgun Gothic']


In [ ]:
df = pd.read_csv('data/22_열처리.csv')
print(df.shape)

FileNotFoundError: [Errno 2] No such file or directory: '22_열처리.csv'

## 실습 1. 시간 인덱스와 1분 리샘플링 재확인
시각 글자를 시간 인덱스로 만들고 1분 평균으로 복기

목표
- 시각 글자를 시간 인덱스로 만들고 1분 평균으로 리샘플링을 복기

단계
- 데이터를 불러와 시각 열을 시간 인덱스로 올리고 정렬
- 두 센서를 1분 단위 평균으로 묶기
- 1분 칸 중 값이 빈 칸이 있는지 확인

예상 결과
- 1분 평균 표가 만들어지고, 누락 때문에 빈 칸이 1개

In [ ]:
# [1분 단위 다운샘플링 집계 및 결측률 축소 검증]
# 1. df.resample('1min').mean(): 매 10초급 데이터를 1분 바구니 단위로 결합하여 평균을 냅니다.
# 2. 데이터 주기를 거칠게 축소함으로써 국소적으로 비어있던 측정 주기가 뭉뚱그려져 최종 결측 수가 1개로 급감합니다.
# * 매시간 발생하는 노이즈가 과도하거나 일시적 무선 단절이 있는 상태일 때,
#   분석 주기를 다운샘플링하여 상위 스케일로 올리면 통계적 왜곡을 줄이면서 정상적인 추세 분석이 가능해집니다.
# * `head(3)`는 상위 3행만 요약하여 빠르게 데이터 모양을 모니터링하기 위해 사용합니다.

df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.set_index('timestamp').sort_index()
one_min = df[['제어출력', '소입로온도']].resample('1min').mean()

print(one_min.head(3).round(2))
print('1분 평균 빈 칸:', int(one_min['제어출력'].isna().sum()))

## 실습 2. 측정 간격 분포로 주기 진단
이웃 간 시각 차이로 정상 주기와 벌어진 구간 찾기

목표
- 연속된 두 측정 시각의 간격을 구해 정상 주기와 벌어진 구간을 찾기

단계
- 시간 인덱스의 이웃 간 차이를 초 단위로 구하기
- 간격 값의 분포를 세어 가장 흔한 정상 주기를 확인
- 가장 크게 벌어진 간격을 확인

예상 결과
- 대부분 10초(정상 주기), 가장 크게 벌어진 간격은 80초

In [ ]:
# [측정 간격 차분 계산을 통한 설비 데이터 수집 주기성 진단]
# 1. df.index.to_series().diff(): 인덱스 날짜시간 간격의 차이(시간 델타)를 계산합니다.
# 2. .dt.total_seconds(): 시간 격차 데이터(TimeDelta)를 정수/실수 초(Seconds) 단위로 일괄 환산합니다.
# 3. value_counts().sort_index(): 어떤 격차(초)가 가장 흔한지 빈도수를 집계하고, 간격 크기순으로 나열합니다.
# * 데이터의 90% 이상(169개)은 원래 목표인 10초 주기로 잘 들어왔으나, 일부는 3초, 5초, 심지어 최대 80초까지
#   기록이 끊긴 적이 있음을 파악하여 시스템 통신 신뢰도를 정량 수치로 진단합니다.
# * 인덱스는 시리즈 객체가 아니므로 `to_series()`를 사용해 변환을 거쳐야만
#   `.dt` 및 `.diff` 시계열 보조 연산자를 에러 없이 적용할 수 있습니다.
# =============================================================================
gaps = df.index.to_series().diff().dt.total_seconds()
print(gaps.value_counts().sort_index().head().to_dict())
print('최대 간격(초):', gaps.max())

## 실습 3. asfreq로 정규화 후 NaN 확인
10초 격자에 값을 올려 빠진 시점을 빈 값으로 드러내기

목표
- 10초 격자에 값을 올려 놓아 빠진 시점을 빈 값으로 드러내기

단계
- 두 센서를 10초 격자로 정규화하기
- 정규화 후 총 행 수와 빈 값 개수를 세기
- 앞값 채움 옵션을 주면 빈 값이 채워짐을 확인

예상 결과
- 정규화 후 200행, 두 센서 각각 22칸 빈 값; 앞값 채움 시 0

In [ ]:
# [엄격한 주기 격자 변환 시 발생하는 결측의 직전값 보정]
# 1. df[cols].asfreq('10s'): 수치들을 정확히 10초 단위 정규 그리드로 재배열하여 누락분을 명시적 NaN(22개)으로 발굴합니다.
# 2. asfreq(..., method='ffill'): 이전의 정상 측정값을 다음 누락 지점의 값으로 채워 넣음(Forward Fill)으로써
#    결측이 없는 200행 데이터로 클리닝합니다.
# * 실시간 가공에서 직전 온도를 알면 현재도 유사할 것으로 상정하는 직전값 전방 대체는
#   제조업 현장에서 결측치를 매핑하는 가장 직관적이고 널리 사용되는 기본 기법입니다.
# * `ffill`과 `pad`는 동일한 동작을 수행하며, 만약 첫 번째 행 자체가 NaN일 경우에는
#   앞선 정상값이 존재하지 않으므로 여전히 NaN으로 남게 됨을 인지하고 있어야 합니다.

cols = ['제어출력', '소입로온도']
norm = df[cols].asfreq('10s')

print('총 행:', len(norm))                                  # 200
print(norm.isna().sum().to_dict())                          # {'제어출력':22,'소입로온도':22}

filled = df[cols].asfreq('10s', method='ffill')

print('ffill 후:', filled.isna().sum().to_dict())

## 실습 4. 리샘플링 후 결측 위치·개수 파악
결측의 개수·비율과 빈 시각 위치를 파악

목표
- 정규화 후 결측의 개수·비율과 빈 시각 위치를 파악

단계
- 정규화 데이터에서 결측 개수와 비율을 구하기
- 빈 값이 있는 시각을 뽑아 개수를 세기
- 빈 시각의 첫 구간을 확인

예상 결과
- 결측 22칸(11퍼센트), 첫 빈 시각은 9시 2분 30초

In [ ]:
# [결측 분포의 정량 진단 및 최초 누락 발생 위치 확보]
# 1. isna().mean() * 100: 각 컬럼별 결측치 비율을 백분율(%)로 구하여 유실 정도를 평가합니다 (11%).
# 2. norm[norm['제어출력'].isna()].index: 제어출력 변수가 결측인 행의 시간 인덱스 목록만 추출합니다.
# * 단순 결측 개수를 넘어 전체 가동 기록 중 유실 비중이 몇 %인지(11%),
#   그리고 최초로 데이터가 유실된 특정 시점(09:02:30)을 탐지하여, 당시 오프라인 정비나 전력 다운 등의 사유와 대조합니다.
# * 결측 인덱스 리스트가 비어있을 때 `miss[0]`을 조회하면 IndexError가 발생하므로,
#   먼저 `len(miss) > 0`인지를 미리 체크하는 코딩 패턴이 견고한 예외처리에 도움을 줍니다.

norm = df[['제어출력', '소입로온도']].asfreq('10s')
print('결측 개수:', norm.isna().sum().to_dict())
print('결측 비율(%):', (norm.isna().mean() * 100).round(1).to_dict())

miss = norm[norm['제어출력'].isna()].index
print('빈 시각 개수:', len(miss), '/ 첫 구간:', miss[0])

## 실습 5. asfreq와 resample 결측 비교
같은 정규화라도 결측 수가 왜 다른지 비교

목표
- 같은 10초 정규화라도 asfreq와 resample의 결측이 왜 다른지 비교

단계
- 같은 데이터에 격자 올리기와 구간 묶기를 각각 적용
- 두 결과를 나란히 두고 빈 칸 수를 비교
- 1분 다운샘플링으로 결측이 크게 줄어듦을 확인

예상 결과
- 격자 올리기 22칸, 구간 묶기 16칸; 1분 다운샘플링은 1칸

In [ ]:
# [격자 배정(asfreq)과 집계(resample)의 차이점 결합 비교]
# 1. 두 기법으로 각각 변환한 데이터프레임의 컬럼명을 유니크하게 변경한 뒤 `.join()`을 통해 열 방향 결합합니다.
# 2. 동일한 시간 격자에 대조된 결과를 확인하여, 한 구간 안에 들어온 불규칙 다중 신호가 resample 시에는
#    어떻게 병합 요약되고 NaN 개수(22개 vs 16개)를 줄였는지 분석합니다.
# * 데이터 손실 강도와 주기의 정교함을 저울질하여, 정확한 10초 스냅샷이 필요한 제어 로직 분석에는 asfreq를,
#   전체 에너지 흐름 관점의 일관성이 중요할 때는 resample 집계를 선택하는 의사결정을 내릴 수 있습니다.
# * `join()`은 인덱스가 동일한 날짜 기준일 때 가로로 결합해 주는 매우 유용한 판다스 함수입니다.

a = df[['제어출력']].asfreq('10s').rename(columns={'제어출력': 'asfreq'})
r = df[['제어출력']].resample('10s').mean().rename(columns={'제어출력': 'resample'})
print(a.join(r).head(8).round(3))
print('asfreq:', int(a['asfreq'].isna().sum()), '/ resample:', int(r['resample'].isna().sum()))

down = df[['제어출력']].resample('1min').mean()
print('1분 다운샘플링:', int(down['제어출력'].isna().sum()))

## 실습 6. 불규칙 주기 정규화 종합 리포트
측정 기간·결측·최장 연속 결측을 담은 진단 리포트

목표
- 측정 기간·시점 수·결측·가장 긴 연속 결측을 담은 시간축 진단 리포트를 작성

단계
- 정규화 데이터의 측정 기간과 시점 수를 출력
- 결측 개수와 비율을 정리
- 빈 값이 연속으로 이어진 가장 긴 길이를 구하기

예상 결과
- 200시점·결측 22칸(11퍼센트), 가장 긴 연속 결측은 10칸

In [ ]:
# [최장 연속 결측 구간 검출 알고리즘 및 격자 결측 최종 리포트 출력]
# 1. `isna != isna.shift()`: 결측 여부 신호가 직전 시점과 바뀌는 변화점(경계)을 판단합니다.
# 2. `.cumsum()`: 누적 합계를 구해 서로 다른 결측/비결측 덩어리(Group)를 분류하는 새로운 번호를 매깁니다.
# 3. `isna.groupby(groups).sum().max()`: 뭉쳐있는 결측 구간 중 '가장 길게 연속으로 누락된 개수'를 정량 검출합니다. (최대 10칸 연속 누락)
# * 간헐적 결측은 단기 보간이 되지만, 연속 10칸(100초) 누락과 같이 거대 공백이 생긴 구간은
#   인접 데이터 기반 추정이 매우 부정확하므로, 보간 불가 대상(무효 처리 구간)으로 선포하여 오판을 방지해야 합니다.
# * `isna`는 불리언(True/False) 시리즈로, sum() 연산 시 True를 1로, False를 0으로 해석해 동작합니다.

norm = df[['제어출력', '소입로온도']].asfreq('10s')
print('측정 기간:', norm.index.min(), '~', norm.index.max())
print('시점 수:', len(norm), '/ 결측:', norm.isna().sum().to_dict())
print('결측 비율(%):', (norm.isna().mean() * 100).round(1).to_dict())

isna = norm['제어출력'].isna()
groups = (isna != isna.shift()).cumsum()
print('가장 긴 연속 결측(칸):', int(isna.groupby(groups).sum().max()))   # 10

# 결측 위치 그래프(검증용 savefig)
plt.figure(figsize=(10, 3))
plt.plot(norm.index, norm['제어출력'], marker='.')
plt.title('제어출력 after 10s asfreq (gaps = missing)')
plt.tight_layout()
plt.show()
